# RTC-SDD pretrained backbone benchmark
**Only official Train/Dev**; frozen encoders + identical linear probe. Never touch Progress/Final. Enable Kaggle Internet & GPU (T4).

In [ ]:
import sys, subprocess, json, csv, zipfile
from pathlib import Path
ROOT = Path("/kaggle/working/RTC-SDD-Research-main")
# Upload RTC-SDD-benchmark-branch-ready.zip as a private Kaggle Notebook input dataset.
ZIP_CANDIDATES = list(Path("/kaggle/input").rglob("RTC-SDD-benchmark-branch-ready.zip"))
if not (ROOT / "scripts/benchmark_pretrained_backbones.py").exists():
    assert ZIP_CANDIDATES, "Attach the benchmark branch-ready ZIP as a Kaggle input"
    with zipfile.ZipFile(ZIP_CANDIDATES[0]) as archive:
        archive.extractall("/kaggle/working")
assert (ROOT / "scripts/benchmark_pretrained_backbones.py").exists()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44,<5", "scikit-learn>=1.4,<2", "huggingface_hub>=0.30,<1"], check=True)
print("Ready:", ROOT)


In [ ]:
# Replace with the exact path shown in Kaggle /kaggle/input for your private dataset.
DATA_ROOT = Path("/kaggle/input/rtcfake-packed-private")
assert (DATA_ROOT / "train_label.txt").exists(), f"Set DATA_ROOT to the folder containing train_label.txt: {DATA_ROOT}"
assert (DATA_ROOT / "dev_label.txt").exists()
subprocess.run([sys.executable, "-m", "rtcsdd", "preflight", "--dataset-root", str(DATA_ROOT), "--require-audit-counts"], check=True)

In [ ]:
MODELS = [
    "facebook/wav2vec2-base",
    "facebook/hubert-base-ls960",
    "microsoft/wavlm-base-plus",
    "facebook/data2vec-audio-base",
    "facebook/wav2vec2-xls-r-300m",
]
TRAIN_N, DEV_N, BATCH = 3200, 1600, 4
OUT = Path("/kaggle/working/benchmark_results")
OUT.mkdir(parents=True, exist_ok=True)
for model in MODELS:
    out = OUT / f"{model.replace('/', '__')}.json"
    if out.exists():
        print("Existing result, skipping", out)
        continue
    print("Benchmarking", model, flush=True)
    subprocess.run([sys.executable, str(ROOT / "scripts/benchmark_pretrained_backbones.py"),
        "--data-root", str(DATA_ROOT), "--model", model, "--train-samples", str(TRAIN_N),
        "--dev-samples", str(DEV_N), "--batch-size", str(BATCH), "--output", str(out)], check=True)

In [ ]:
rows=[]
for f in sorted(OUT.glob("*.json")):
    r=json.loads(f.read_text())
    rows.append({"model":r["checkpoint"],"revision":r["checkpoint_commit"],
      "online_dev_macro_f1":r["metrics"]["view/online"]["macro_f1"],
      "offline_dev_macro_f1":r["metrics"]["view/offline"]["macro_f1"],
      "online_spoof_recall":r["metrics"]["view/online"]["spoof"]["recall"],
      "runtime_s":r["seconds"]["total"], "parameters":r["encoder_parameters"],
      "train_samples":r["train_count"],"dev_samples":r["dev_count"]})
rows.sort(key=lambda x:x["online_dev_macro_f1"],reverse=True)
if rows:
    with (OUT / "summary.csv").open("w",newline="") as w:
        writer=csv.DictWriter(w,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
for row in rows: print(row)
print("Results:",OUT)